In [185]:
round = 2
file1 = f"{round}/annotated_data/conversations_{round}_ali.json"
file2 = f"{round}/annotated_data/conversations_{round}_wilder_annotated.json"


file1 = f"{round}/annotated_data/conversations_1_ali.json"
file2 = f"{round}/annotated_data/conversations_1_wilder_annotated.json"

#file1 = f"{round}/annotated_data_redo/conversations_{round}_ali.json"
#file2 = f"{round}/annotated_data_redo/conversations_{round}_wilder_annotated (1).json"

labels = ["Recollections","Expansion","Refinement","Follow-up","UI","UR","IR","Error","HM","GD","CM","RD","AR","Context Memory","Anaphora Resolution","Separate Input","Content Confusion","Content Rephrasing","Format Rephrasing","Self-correction","Self-affirmation","Instruction Clarification","Proactive Interaction"]
labels = ["Recollections","Expansion","Refinement","Follow-up","UI","UR","IR","Error","HM","GD","CM","RD","AR","Anaphora Resolution","Separate Input","Content Confusion","Content Rephrasing","Format Rephrasing","Self-correction","Self-affirmation","Instruction Clarification","Proactive Interaction"]

import json
with open(file1, 'r', encoding='utf-8') as file:
        data_a = json.load(file)
with open(file2, 'r', encoding='utf-8') as file:
        data_w = json.load(file)        

In [186]:
def extract_turns(data):
    turns = []
    for conv_id, turn_list in data.items():
        for turn in turn_list:
            turn["id"] = conv_id
            turns.append(turn)
    return turns
turns_a = extract_turns(data_a)[50:]
turns_w = extract_turns(data_w)[50:]
assert len(turns_a) == len(turns_w), (f"Turn count mismatch: ali={len(turns_a)}, wilder={len(turns_w)}")

def to_binary(turn, label):
    return 1 if turn.get(label, "") != "" else 0
ann_a = {label: [to_binary(t, label) for t in turns_a] for label in labels}
ann_w = {label: [to_binary(t, label) for t in turns_w] for label in labels}

In [187]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
import krippendorff

print(f"{'Label':<30}  {'κ':>6}  {'%agree':>7}  {'#yes_a':>6}  {'#yes_w':>6}    {'#diff':>6}")
print("-" * 75)

kappas = {}
for label in labels:
    a = np.array(ann_a[label])
    w = np.array(ann_w[label])
    pct_agree = np.mean(a == w) * 100
    n_diff = np.sum(a != w)

    if len(np.unique(np.concatenate([a, w]))) == 1:
        k = float('nan')
    else:
        try:
            k = cohen_kappa_score(a, w)
        except ValueError:
            k = float('nan')

    kappas[label] = k
    print(f"{label:<30}  {k:>6.3f}  {pct_agree:>6.1f}%  {a.sum():>6}  {w.sum():>6}  {n_diff:>6}")

Label                                κ   %agree  #yes_a  #yes_w     #diff
---------------------------------------------------------------------------
Recollections                    0.772    92.6%      11      11       4
Expansion                        0.666    83.3%      25      26       9
Refinement                       0.615    87.0%      10      13       7
Follow-up                        0.884    94.4%      33      32       3
UI                               1.000   100.0%       2       2       0
UR                                 nan   100.0%       0       0       0
IR                                 nan   100.0%       0       0       0
Error                              nan   100.0%       0       0       0
HM                                 nan   100.0%       0       0       0
GD                                 nan   100.0%      54      54       0
CM                               0.791    98.1%       3       2       1
RD                                 nan   100.0%       0   

In [188]:
len(turns_a)

54

In [189]:
#https://www.cs.columbia.edu/nlp/papers/2006/passonneau_06.pdf  
# https://stackoverflow.com/questions/52272901/multi-label-annotator-agreement-with-cohen-kappa
from nltk.metrics.agreement import AnnotationTask
from nltk.metrics.distance import masi_distance

def to_label_set(turn_dict):
    return frozenset(label for label, val in turn_dict.items() if val != "")

annotation_data = []
for i, (ta, tw) in enumerate(zip(turns_a, turns_w)):
    annotation_data.append(('A', str(i), to_label_set(ta)))
    annotation_data.append(('W', str(i), to_label_set(tw)))

task = AnnotationTask(data=annotation_data, distance=masi_distance)
alpha = task.alpha()
print(f"Krippendorff's Alpha (MASI distance): {alpha:.4f}")

Krippendorff's Alpha (MASI distance): 0.6198


In [190]:
x = 0
for i, (turn_a, turn_w) in enumerate(zip(turns_a, turns_w)):
    diff = {lbl: (turn_a[lbl], turn_w[lbl]) for lbl in labels if turn_a[lbl] != turn_w[lbl]}
    if diff:
        print('===============================')
        x += 1
        print(f"UUID {turn_a['id']}:")
        print(f"Turn {i+50}:")
        print(f"  user_message: {turn_a['user_message']}")
        print(f"  bot_reply: {turn_w['bot_reply']}")
        print(f"  Label differences (A vs W):")
        for lbl, (val_a, val_w) in diff.items():
            print(f"    {lbl}: A={val_a}, W={val_w}")
    print()
print(x)


UUID b50b6c66-2252-47cf-a890-9f13757be68c:
Turn 51:
  user_message: can you show me the exact code line?
  bot_reply: Here are the exact code lines:

- Record failures and timestamp in DB: `AuthDAO.recordLoginFailure` at lines 294-305 in `src/main/edu/ncsu/csc/itrust/model/old/dao/mysql/AuthDAO.java`.
- Read failures with timeout reset: `AuthDAO.getLoginFailures` at lines 379-405 in the same file.
- Action wrapper calling the DAO: `LoginFailureAction.recordLoginFailure` at lines 47-53 in `src/main/edu/ncsu/csc/itrust/action/LoginFailureAction.java`.
- JSP hook that records and logs each failed attempt: lines 66-75 in `WebRoot/login.jsp`, calling `loginFailureAction.recordLoginFailure()` and `loggingAction.logEvent(TransactionType.LOGIN_FAILURE, ...)`.
  Label differences (A vs W):
    Self-affirmation: A=y, W=


UUID b50b6c66-2252-47cf-a890-9f13757be68c:
Turn 53:
  user_message: from 1 to 10, what is the satisfaction level you give and why
  bot_reply: I’d rate it 4/10. We do log and 